# Agent Mots Tus — tout-en-un
1. Règle `REPO` (ton dépôt GitHub) et la voix en haut de la cellule.
2. Appuie **une seule fois** sur ▶ et laisse tourner (≈5 min). Les vidéos se téléchargent toutes seules.

In [ ]:
# ======== RÉGLAGES : modifie ici, puis appuie sur ▶ UNE seule fois ========
REPO = ""          # ex: "https://github.com/TON_PSEUDO/mon-repo"  (vide = tu envoies les fichiers à la main)
VOIX = "fr-FR-HenriNeural"   # voix d'homme : fr-FR-HenriNeural | fr-FR-RemyMultilingualNeural | fr-CA-AntoineNeural | fr-CA-JeanNeural | fr-BE-GerardNeural
GRAVE = 3          # 0 = normal, 2 à 4 = voix profonde
VITESSE = 0.9      # 0.85 = très lent, 1.0 = normal
CHEVALIER = "center"   # position du chevalier : center | left | right
FORMAT = "16:9"        # "16:9" (comme tes vidéos) ou "9:16" (la même vidéo centrée sur fond noir vertical)
FONDS = "fichiers"   # "fichiers" = tes fonds du dépôt | "genere" = l'agent crée les fonds par IA | "mixte" = les deux
NB_IMAGES = 8        # images générées par vidéo (mode genere / mixte)
MOTEUR_IMAGES = "pollinations"   # pollinations (gratuit, rapide) | sdxl (GPU T4, plus lent, en secours)
ACCENT = "rose"        # mot en couleur dans les sous-titres : rose | rouge | aucun
CLONER = False     # True = clone la voix du fichier ma_voix.* (TA voix ou une personne d'accord)
EXPRESSIVITE = 0.6 # clonage : 0.4 calme, 0.8 très expressif
# Réglages par vidéo (facultatif) : lignes @ au début d'un texte dans textes.txt, par ex.
#   @chevalier: chevalier2.png   @musique: musique3.mp3 (ou aucune)   @voix: fr-CA-JeanNeural   @grave: 4   @vitesse: 0.88
#   @fonds: genere   @style: (style des images, en anglais)   @scenes: lac au coucher du soleil; château dans la brume (en anglais)
#   @position: right   @taille: 0.9 (taille du chevalier)   @accent: rouge   @un_en_chiffre: non
# Sans réglage, chevaliers et musiques tournent automatiquement d'une vidéo à l'autre.
TEXTES_PAR_DEFAUT = """Avant toi, la pluie n'était qu'un bruit.
Maintenant, elle me parle de toi.
---
Il y a des mots qu'on garde au fond de soi,
parce que les dire les rendrait trop vrais.
"""
# =========================================================================
import os, glob, shutil, gzip, base64
from google.colab import files
def sh(c): get_ipython().system(c)

print("1/5 Installation…")
sh("pip -q install soundfile faster-whisper rembg onnxruntime edge-tts pillow")
sh("apt-get -qq install -y fonts-playfair-display fonts-dejavu > /dev/null")
if CLONER:
    sh("pip -q install chatterbox-tts")
if FONDS != "fichiers" and MOTEUR_IMAGES == "sdxl":
    sh("pip -q install diffusers transformers accelerate safetensors")

WORK = "/content/travail"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK + "/fonds")

print("2/5 Récupération de tes fichiers…")
if REPO:
    shutil.rmtree("/content/depot", ignore_errors=True)
    sh(f"git clone --depth 1 {REPO} /content/depot")
    sh(f"cp -r /content/depot/. {WORK}/ && rm -rf {WORK}/.git")
    os.chdir(WORK)
else:
    os.chdir(WORK)
    print("Envoie l'image du chevalier :")
    for i, n in enumerate(files.upload()): shutil.move(n, f"chevalier{i + 1}" + os.path.splitext(n)[1])
    print("Envoie tes arrière-plans (plusieurs possibles) :")
    for n in files.upload(): shutil.move(n, "fonds/" + n)
    print("Musique (facultatif, Annuler pour passer) :")
    try:
        for n in files.upload(): shutil.move(n, "musique_" + n)
    except Exception: pass

B = """
H4sIACIhAmsC/61965LbxrXu/3kKGN7ygCOSQ87NCh06kS05VkW+lKTESXHmsEASJOEBARoA52KGVfs1zvm1f+2K9iucf0dvkic5
37e6G2iQ4EiOrcRDsNG9evXq1avXrZsff3S8ytLjURgfB/GNs7zP50l8euC67sHTWRDnjvtNkmfOm1XmOj0nD+7ywGl97twk4Z2T
TKfyHE7evU2cmyDNw7EfBU6386Rz1/3dScdZpu/+Bw3e/ZezTLI8SNsHB8/vgsUSlXoHju7N8dlRe3nvtFrZOA2X6E86ytr5XY7C
0cyZJvEkO8bzdRzO5rkzngc3fhQC4jKeOZeAxX+tVhDPwjhwgskMaLaA5Thwpmnrq1etr4M4Db8NVqkf4c1ilYVjh39/WgXtxfL0
4MDqsuesYj3WpZ86oygZN+Vv5mTv3qLo3dtM3qzQVxTO8HecxHkQ+zHxbQn5wsUySXPHT2eomQXF9+w+HoeJ+brw87l5TjLzlPrx
JFkU34rG2XyVh1HxbTVapsk4yIp22X3xmIPM0zAqmubhonhexeE4mQQTP/cPpmmywFjyeRSOHP3+eyJ18EPT+brpfPX9a6fvcDab
MrFN57QDUn/sdC96v8OwF4sAfWWaC7KDF9/8afj8b2/QZu22f1zO3KbDz0A9YLbk8zYYLd3NwV9fPCsrL5Zn8m6R3KjP6xtTd4G6
L58/fYaKnfbJObvPMLYYs+vfkOiRr1jSyxoHb56+eCkVnziVimDGf2aVmgcHH3O6ft0/h1MS5n6YBtnBJJg66Sr2xotJoydcmQKV
cqLafDnI8tQbN8DUqTN2wthB7aumM/aX+SoNhskqX67y/pt0FTSFC+WxIdBCgG+nAerFnEHnI4yzp7kffflhFjivVjEn+3maJqlX
vOM/90tMF1gLK/LteJ6s3r0Neo7rPMZ/mKMkjL0azBqD3mmnc8ValzErp+0snwRpOmh1zzud3lXRhUJRYadqYSggMokywcLLwyT2
yGqaNHgL4ghF3OkUFBoFnPCWTHtA9OVrNk9uh5AQaRhkLAByWDV9A9FtVsbIYbaSKSuOs5v+st/BIzu9qqA3jRI/94AB0ITI8RoN
jWiUzLxFNtMoLtMwzvm9iRarbK6n4rdgnFKGem/evG5I59kSnDTMAsoSsIvH2W9CwizCvH9y2tFIQbq8xIoTwQOJ5McQRkk8zldC
D2caxpTCGSQSRjtLk9USE505HiicJZMwcGKfjBZFQcP5TCBiUSz9FXgnTxYjiE8B4oBPlvPUz8CGyYryPXO6EOpjP/XH+bt/gt/b
FHQCICQqfWcQGXoKD0XkoTRoy7i8FAz02FU8XZCdLF00UrwEeZlnWA8rrp0BVobupIAovZVsDwisC8Z2sNQ9PDfApnyMGs7nxLlX
YRGB3/aXyyCeSO3KW9Wt7rIsYEW1Ugi7YTC2keBAAz8dzzHSQfujP/zrP//7//3f3tV/uDKWxr+NhBpf2by+qVpNg6uCVD+RVFK3
bMry5dakeH/4fV+je9W4zDhDP20jC8jLnSHz3+0c4lVovSSthVV72wsS6OcCIQVbTTy3iS46mq8bO5VluLnze+eiswupBprzIDQu
cU2q5aDHpo+d7lW56rfry1AHul7vqnbQwHDZe7CniqxRUvCb7968Hr749s3zV89evOGe6rkjbMAhliGF1WwFiSdPwTKQTz9drFRJ
mtz7q4V+DEW0YZ36ea5eF8qQa4QYtZE0iYKhKDEiRkrZ8TSCpkZ5EB/6qVLQlj7ESQB848mqAfWne3Le6p6dO4uEC9FfjaEP4RmM
AyVuEubFumcFyvCgzbnwo4isf3l7+K///N+tK7PWFTGgdxXsCQLGAIdunN/3hXsIqMEv6LYkrR8Zik7ddVFto7r1xuGI2qbGdZlg
zVyctj49cbKGq/oc+cQuC8DmFQz91s+CnUzu0B+PIXGVtG1HyW2QgjEaBsKEaC9k5Sy4crbmEUORYnRVjA2N6sbgCtqGhpmj996m
2XzRrFFhHD/S8zm8TcM8GN76N954voqvMSlZSqmcs6yphLeeYa3DxasFNGqMP17axVmyiifUC/kqm8oroyH1Ubf9c4BdwuOul6XO
kYbcdCb5/TLou7Jtnp5o8i7DYKzEfil1lNogSFoSSyoaOqAXPwPr+ffeeBcy1Jts7i8Dr9W1lmcVgsZYvc6mbSGPKC+aJMAYvWAZ
jH2q5XipADRIOK6S4djHsqHiudE0zvNsOJ5jUQXpKLnTW29BYajxWZXAeQJJr4ZNFbps2l7k0TDnTKuKXxZvvllF1BXj2cqPsOfr
FTQJuDIUQu0ZeNUdj9xC3VPvw8z5NomDkqKT4IYbxHg18V1WE2za/NoOsyGU4jDysTqwDwcR9nV3vFyVO5rpcy9mbQ5puAQbQqWM
g4mH7mBK9fFRTonCeEBsrwBMgMpLNftVtiDf16o31k5zfYs2k3Cce5FPTGbBMJyAM6gGBnf+bBYoja/PyWgHd8Avy8KbML/HBjud
DW8D2ob9Tvussi1L7TSYDsUcrMrt69uB668mYYLBJotlPqSmKMOptioagRnMWDFXMTEC4zWdo6Pr27JXRQHDrWjTzmBnBj9jPtqY
CfyV5Wn2n5rlrXqoLHLBSK1Hi2Vp6r6fWVmLPKlYbnmKMdDIK5ZL+zbM58NsNZ2Gdx5NsVPNgRwfKkP8UiR4gkOGDWritJxuAwKi
24Fa+niyeaQVQT/j5kxmLeZ+KAoHpjTwTq3pztP76mTQmERbg2ub1gpNVT83I5TuZUKaglmffyD+wnw8V1whj9XdWlvcYnqxi3bm
3yhhgWE2tlSAbfzNv1Ea+NdFSXA3Dpa581w+qHNDmG5xlgZUhUK7rJ1FQbD0TooVzpoP2nBTlyRpUaT4heXWc9ZsuDHzpK2ohba0
W/d1phR1Bwyaj76UnJx1Oh155Y/50XULNroyXDYButh6hbOyAEoWaJOV6sSrgHq3tqqXMJKcWQoCK7sEUhHTnsp7sHiWBYXqMAV1
TrBuHK9VQHWOsZ2rAeWLpWHSkkFjfxF4LhFqo/AXD12GgKFObZtx6sLCyYWbhBpH62mvfTbdNH2IF5++KlXe9OlSSfrrLrDUVUqR
iq5+guwMsX32p/1ut9PL+z/1bvvd3qx/2rTfXdBoLl+2TqklLZbaPFUeHvpAFIviTUP0FKGCmZKFfx0owfTA0i8rDVP/trbigS0i
SVZ7g5FZr0ofFu7isBd8rwJf1aXuYXH7B8/dFogdFj7YcgHU8XPRq6hX8nXtlhu329tSAogEVp4u5+NmoPYecTNeld6VWuL+Wi8B
xQW2ZExFJhqvkP02SSdDyobMk41Lq9iKpBWRKmrJ1KffdQgzDVI7NdvBD+rrN8WWvU8T0e3chgM5brfy3GwBZZqmLfbNFXYvpcZh
l3ji7uoIBk5VURCGD2bY7obFpoqVGNMLPFL8L2PECjAagVYHSirkWKKZcpbVGovbxmUc5qEf6e2+T9oNeuAgeBlu/AlZC+SyHG6y
56MzUWe82zafjU0INPAIA5sPkIINS9PBqOTbrXyTZtkuflgaVZBXtuIijaq7ijYMSnD1OxH9nmkAbQs2Rioe6zyUtxg2GQCPMf0/
BWx6vaau43xkJhloT/CQxGJjeWvORQBX4JPO1QYD15wJBSdN7kK448KpnvU8EdVP+XjEw2DKc7EAF/6dVzgD1exClei0sSl12x1V
Vylyyp8EGZTTnXMi5KQJKl0oQol1t1p4ukHhCAGC2nXUaXcKPQTtMFMzgfEzqE04TdOZpZjQ4lPoHkn1Y6ecOsvI9wSe9PUYbY7Q
1+8sbQKl0GZ3/QC/UibAgMtamE1sTs7T169FJvhZJovBK+SAJjSQ61TdnkqPw5iOndMLqG6bni54JN9ZfgGFrnMywRuWwgnTOW+f
TDeulvsIxeTDqb8Io3tPnqky96kwlTrB90nEyAsFl41vD+siBWNDQYD5/n3k30/hNneegdHwDDsUM4MFFUXwVzYKRSHUfbKfB7RH
43dWO8oYoSQ/lt2jpRzGfH60Vohv8KWAedXQjAq/VGPQ2XX5bK+xbUdelhUsBvDkLs/dHhx7f5mkPj+fBT/6f105r0kL9yGFeJZY
DvJxC4MYz2VI053BoF+4EnqjJJq4VzveKgAyjg0lsTKqUx5alcVqrWL8jV3XluEef/FLiaJbVscMTnr65ZfPvxU32Bo+rYx7rPvp
k5NnX32lvFwr2Xbdk2emSJxQKCKjbSSwM05WEeJ5sgwc74s/vTKayWgVRpMhMPBETir3v9AI1IIyBUUMW7Q4ffqq8ybjk8M7mI7n
TUTHEG2AFhBOp2mgtgI1OHGlVxzTloErgn5LYkOTSI2IoiC7I3XpV5QWd+IpsT2/liPbKzzZn/edU+6+Cpjya98KoM+dkwt5s2Is
5gpl+OhdEayr/LnNz3pu7a5IhAfdK0he0/jkCvBgOp9vTb8a9Hsc1YMr21te2L2NWvd1DUTlNkiihMA0a4gComapacoGaro0g88D
cc6VWpg7eC3xY+dFPE2uLmP17Q0Uk55zc9budB5fxlyYr4Lsbz3n0aT49nf17YfUX77O76GgOieXMaJdFuS/nj125F0GwF+JRAEz
wh5BiBSSJC6eFHt9n2JXTO+/xJhWMOJfB/BFTawCC/R3q5wBDfPmC398XTxjOTedFxCJITxlf0HkLmVVwIOIug6+4073mjH3v+nP
v1cgv176Y2zTIF88o8r8BdgzSGUUTUd3i4Zzf5LcohIDSgsh9zdQYMP4pXl4VYGqyv7adJ4zBgn4FUJp+n0Dv1HzUdZ8NGl+8nWn
85X8235U//D4hXlsdZsd+R/cCvKfBVmKmyftk+ZZ87z5aYf/R8n2RD2/oTPXmqSX/n3AKVCa2vN40nQ0CdT07Q62GKEF9/l0GozR
/g0UG8ZBHzmeCpDbYkXxZaMaEitlRNh0ZlydAbw/ynOk1oK15EQ6o9WMq7nLqCvj3sXr+E5eSqtBKKGMSkUuNilG+ISCQsNXTsCZ
Wei67mOs9/NSoMdcTFDrvJi2TF3li8Z2begYAQmq0H4s0XlLBUpsN2BFDRs2lVtoVpU2eCfK13Wb6qsP+9ZdU/qLW1gXbFTB9h5X
Eds9FfrnNDOMaSUo0OuOjQRKHT4ClwkssBJ3hKRChFGy1QgRg//lreJ/oH7D+0N/0Gx/1lPhsqP/aBgrU0Y1jfxZ1kerF43tYRUC
D88NS++GeyQz6prykYnKiwYN6NnXwX0/8hejie9c9xzzZnB9ZbvGuZ/vkvlacKqym4DtbVNOCV7uOhhzv8DJxFOJMPaG0121oBKI
dNeXl+NPvn6UfbJ5lKlntdQ/2chqkV4Epy2XG3nzPbArNJve0fONHvz4/PISG7cHIfNo0ri8nPoT77zTPO/geRSt0m77RPVNFfcH
RjO4y4Os/P61aOxntsouS7YYzzPYiQnCcj1EFynJsqYINS2g8P1RVr+5ssdCJZeVgS6LAhqK4BQ6F0zgR8ZqPJHi8pIsCRXWkBie
J3udSsFQbQTXhi6Cf1zL4r67yqctY4Jr7UtC4VL/l9sfzuzd2/jdW2WwiUaPyE3I6H8L6xEuPu/F08bBs+dfPf3LyzfD12/+/vK5
xDZvQGPY65hHCXhF9+Arsb8Q1oQ+Gk0CGnLACcIjmeZIOIgWNAxAYWcCZ+uCIhXKKT0f6NmWxaWXJ06cZZAsKcvxqJwweLgFr0P2
p+iOWUoMixYIQo94TrVzoGyV/DZx4KOEfc8kNGA5SpHmIbbtzKc14gTwYDhJPodNTKUZ8j5ioAQdY1vOtOPJRddpgFSocQQFxVkt
kdgBZ+0sDTDM4J4+akMH08KHU5oJdhmpgKSKLGdQjlldMfyRDhyAWJMoB1kiuNpMs6JAYSNDBeMi/JvJihViLsIsNw2mSEySNzNQ
k4incN4HSPNIxVOBgTlQTK5NdSpYemzFUDkPJeJLpjhQT1AYhPE1Jswn6Iyjzm7JFexwGmZz00iNEpWgamSqoWaCSBL6pD5DgOLw
WpXooxL8+z5nlY2yeQDiqPQ9bh8KhoKrQJmGESgIzqcrTzX1HXiVRnSRoC9QVk3SGBk4QeCwGOp7ImzJyFlTqOLATYT8MuYK4p2N
E9LlUuhzZhbA1LA3J1EgOGw1LogQUg+TCRAyS9tJcAv6JctM4SANR9ABuXXHRZdcGMV8SMWfVmGQCyc0BTmwbrQKWorlKpSABy5N
+TrJMib7VSd3FGQhtkhfMVXBgOXE+Wys0i6z63tDy+kqithvrDgV2QYCDekg5aLwV/lqQfGDeEDZIe0k0ktTioKaFBkl18G8nDz4
p+4BFVIvn9McLPjlFqKQ4iLAJOxgewsHNfu4BRpgCKzJoBygLB24tEjUq4ODPz//+w/fvXomNqioC8xwmKzKJNds/O6fYFoQ2okO
ARBcmQZ6DUSgPg3UD52X6owEeQIfNtvXk9csVb7NFIErhI2ouqD11iToIQoLKQQQKL8R+WB1Dl2AbZGP6HMmODWg48S/jaFDwT7A
csOShiBbTYoOpfIvbpUhBSWM2EzPHSYrZXhLsKWQulfj0mt7xTDhfdF8SvNehimTjcWKaZ1MxWOhvKyUdZqNlOC9CbGaULEqnI2D
4VesvwXmmSDK/SLlxsBJyhOw88TeJwyHm2nUKCK9BsE9xMXGeltUU3wfFcI9QivBtLqN5MgqU9jJuPSOuQiw/yIYCNuwOsHTYCU0
171rWa8kMvaCDAL8uiL79a6ciP65BQvap0LpNwEXr8J8P9t/mFShA1swohjWInuS0PMuXUFsJEvZE4xgVjtMtgwVMC1ParZkSuvA
F+JhKWEz4lTPzMoGTmWH80Tp1ma9zyECmpUtrdywp2pKK/uNEUqxQnZEcHpfkm1E01C40d56kNAJZiO8X65xwBWbYsZEbO3d5TVz
1Y4EgTFDnjrlRTh0HMEAc/wRd19LfaGw0DARvk4L9kqgohKgEgYKexZmeVUmMAQKpdCaP2E3CvhSKiLm17M1pFKbElj0LGMRBpNM
k9pmUzChgUMHAgFRq+CueVvplUDowFZABAGzIEVTlQ2t3PElUUnmC/Gme00wcssvUYL0QQXD9Pv1Ieg4SUKOz53R/UPa0QRKmsC7
h5TDfklNRQFAqA7b34S5NWC/eCy6MPdj9dqeov3jBneI3IdiyODQWGSungsud1ndmrmR6kBNW89JGuJVIS4Qm1GSO8YAim1W7XVC
W4pBpDQj/xGyP4pocWDWYyVw9erPjd5ZMEogqZOamhTntLRyzoPoiyUuoi/W6IblzgOYJoNsK5vQHADQHnBtt43LLDnrGEY7prOK
KQKe++1Xz+hPKDIRlXOjrMp8mFmS3jNRH/n/7jdxkfeZoesgG6IHHY+P6a0Gzy3sAI04ZFloJeglCYNzEVahp94pA7Jqm1Ol6tck
TZYW9Dzc44q44YCNxtUGhy0yb9cTISKVvZD+N5IjigIC3XUQsNTY6jd2bF/cKYO70se+Zfuhnzsb9JWVe8NjN23kYEyhe3gE1dgm
EVsMejH9Yny/m3pdZkSjGCmtjhUaYUVEBQlqKzBIkHoSYYgOl7Lxic2deSpWbcIYGQRYNblrlaLyqK2OGe2Wp8g+MZhKTsD3OB+j
q7xYYMXIG1QWu32e58usd3wc8k3bxqPth8cKlWPmrdq9tn9aJcyxlLeDHgKJ1RjUYwQe/3AbThAoXP+w+WSuEvXWX28+4Wj6a/7d
fBLDQzRL+jkiLp9IMkAfZy7utCvj1ySWgQJoWiVH+5X69FDclFAC1n1/7f4FIcqWHH2jfPgm+ZlS5fi83XE3VbcV1+IuVHzlhg7e
+Umi5AHmto9zMvRfwjtVBbHj6hndw1PpEfJWlj1FvcxVW6BLG4b6w0WvLn8+XLSF2uC+8+5JfRZ9TaKZK5MO+iVLKgfAaMvNaiXO
/EZpcGcmzbAm681mPivdrVwl2eQueu/qKPN0l+Ey2MpvIYQy11Yq7KTaypqZQKtB5mVaZPW+lp3gmRQDwb+9/B6N6WSzUpalu70V
d5Jsq2e1oCKPeLDsHssuExitiQHSuotaSDgPWkiZEN8zhjhUidQqEVi0iO5FNSXmxk+hOCDmOV12L9AM4xlm0E6x8WVJmm1lvRB9
ZLWw46GsxiEyV4fQXgHaZmST4iOkZH4PG6p4qbjyic6fVKYsWYyJyI02tA1kxA05W55MmZ6xmW6vZ7VvJjeG2ptDMzB5O67a3ixH
H0WHdiZAhwWHYEMNEJlYzajTKi8hNIhxoS5WlhclU/f07IySQGTTpxdPmsyhH4bxNEipww2RSIVMoxPUma1CakYoYsytf4Go8TbI
mRlxf9Zoy7LKEKgxo2SOO/d6CSE1GiojVRZ1yd3isR+WlmhmpdhNwlQlmdVt77oCzk/iL33L3Kf1ccLgDjv8MLm25lrlssmuLblt
XFg6vw1yexCIpA1UVoO9Ihn4UMuHS0eOIm63Vzk6opNQb9hWTqT6aDYslBR9qgB02A2cZeNqKENBs6S+4gyVq5yNN01nrZKVaV9v
XCvfLGDkSm/1/GAwoMvDrUdHVoCL2aeaksz3dDGsdSh5MXKItaLfBLEO682qB9NqNyP+87b3eCEhoCD8UiWyjtxpYdcopZ0WdTVx
E1Bv6xzS/kTmD5TidooYk8TUPrGWSOMGirfvrIH9RmQjUvKZMgaBbXLGcADvamPtJLsBH55XCuPVbj6aU+3qWA4B6anfOLCZkZk2
ZgLf1abdbjdKUU79TkjxYGa1yioJdCc60oHkI8dDpik8VLCBuHKdcdhC5CNbFQeLtOImPfw2x4gRaZFD7uoQKlepvfSntOTMEULm
v8oCWZbn+USV0LXaKXw1I8892tq9ZYtrqyR/Y12wrTmy/Q/HnMem+i0UbDOvwU4ZctuuTlLUJMjojcH5LOJUSQ1GCiSDBDhGxEM+
TZWWh9WEKDqYTA8lQ95BNsdYTISu20YCeBmgk69mTgFpP/blPN9MK0khKsdbSep1dou0tmy+6YFwkN9JGs54WhbJQkvgNZQoVz+M
x1gkOPbqbgEZQzGiAtuD5tq76x964W0LXxvHJ4/Xxcg2R4yhIwO8Wb5uHHknR/nxerLBoabD3jbce4KatwAVdQ+bMJJxPKrfbU6x
36xxAH/T1Gee71c3Zyedpbt15FqJPyobA7eFMNvSBKWRKQYDXASmK0f5WcRMuImK1ZvkalD2qsbeU5QsiKcH/kGks0lVDmg7hjd1
9w6xsV+dv6EAL3JIgfuHpupJw065FWTlRgADmwh4OEjKSPwNz7dMJDH1nDmUsseh8HOhXFdJ5c4u8ZWXamjmoKUmIeNmyW0p60km
5ftm4UOy4o/QpxQyF/BmKvnw8Q5k63xCa9wTALBW7k4u5KqFFk9QBZIZCZfKPXPFpXicSoJh91OVPV+cBEF1qhNDdQmHpzRuUf+G
2BONdNprYoY83bNtxfCQHq8O8dxXf/riaSnAYcJAQ+f5kZjZ5njD78rp6UueIjPjTs47kuzMA6zM82DyuGg8Y+5ADFi/e5vD3xjs
5ybBF0Bxy0hx3waPX1R9EwvJBpFjGeGi8YssoCK1WuPCrWZPdvWFyq7W2xEcvdyLcmZqw83MD3OuFYcTKDRracSXXkFHfrOOyi9U
My5Pj6+M4q1U0HIubRFflP5GF2bAdwpy01en7ssIGD8aIj8nSCwdF9eBGB0XaZXXmrv4+KGarbr+pS9tmO/DE0pybujAzAumpXt8
VtzEoKm7c7pGnzgrj8zcTCTtsZBBUqGa5y5Vitwp3klyUFVsHDbn/Erva9bvtbvTTWbdqbMWaKrY1WmSGu8T4G3lVmvUzVEF65SI
Rt46iJ1lFlGQ34TTuDqTReWyUYe2krxFjWaBVs+ZQyN5u3QASixRhKSbijuU307PKGeCqfp2caFPzSixAb1Xu9xiQNnJ1cVjmVNX
dD3USbv8bnJCBRFdVkn+KpVA6tOdD9AALUqWWuAWwU9BcNHQzBqcZZUD7caaoa2srKVpCITgMqAsXYR39KaUuLD54/6upqfBNPaD
FdswqAWqURUD9D1pO7y+6MVTJLFZO63G6QHj03ANqrhmdULk8CoCT31ztZmnLDpUEsu0opejmw+ZkwrCkOb+jxAL6lCBoSwDFJm5
HAp3lOBekpBZH35l+HqEUEiHE/LWJDDne1FkmAylOgsZqOh8vJOm3M/UHiOKzeMeXNktAdAAETwBaAr0ECXDttYZzBd0B0sHO553
0GTQ2+uJZqVKmrjqhmo6X3EzBKdLIdM2t3LxUaMwCPm8hOzvWGlvCpjtl5bjS1ULvHS1ygBsw7uUJlAVg5kxk3mfU3m4zDYJFKLh
VdPMCazZrkpdfYSDPly0SsOCtlw5KGbZtfbSPMPSXCQq30xuvzFqRIyTabUH0QObBXW1x6LBiSKmql/j+DklndicQXFc0q79Qco2
ND8tx43yVxxoNCIxnqkufd2l6hqa5kEtgiLTVYuFblE4xo3MkJhgOUhTz69A3oFer8AW+ArQK0X86bikrHbzup2e3l1FExOdmKtD
rm3Sn1B8kBdyI89zFTN03J8RtQzjolot38FHU8yExX68QQk6DYzF6oK0NL5SzUe6hGzrxM7KKB2X14wMlJ/5arAON72bqzvC6kt9
OUHXX+fppmc2/v6arzc94AA9ur/Gp0zy4A6tryzJqukzdflCUehjCCwqvlBOeeHSSO7l8nYuVON1LDT3RxCH+IgT3jMGBua5vpwX
MTEGivadJu5AyfSdJekW0+cwa3LQT1JTPBo5nXb3XP2FvumZb6fqTUeVqW9n8qZzbtOcPZQkY/YfVMk+zs/07vHfbT+87c374fxo
DVpJ3m8f6t74Gsee4SeIesqr3D8cBTkStpBh3FznnQ3+dDeNQ3s/eqAbQu9irtHDb9JfHQ+o61mK21lSlW87GM3M1F4b1wV9FfaC
FotZ19lSllzvh9Yt7HzXUpncH45wXV2ndauKteokpSeqdI8KdU1lzf26NX8cTr0I5/ia0MUa+HN6cvT10TK59bqt/Jj6GbwqnYYr
fIcdqbgySORmwLQ33pXjr2ikjFZ5DUlEJHFJKG9A66S3vp4XZns6G/mD6/iqjpqgF18xbI/jNpi/w/X13eYQk4iHezzghjfeSIRA
2SC5MSCgmKpgM3OprfT/y0uS6Ng+EdBjyeVlz3Q+qdydVSiRVkiHXl0FXBxnpYpLzzylE07EKfviF/Vc9D51ewSXoR8Mkt1tDl2j
Pue76CHp54Z3FeHF73oIzqjzC9oKUDct8gY8mgsZ7ApqPCIKils3jeTg2zfh9Rs7Yd502VQzx1scYWk+aS79ifrC2x17nZ6XzFvh
HKzZk9Xj7s4kZgfTAj0ZEgnjwvOGowMXiJTsQ0HrYRrXRlWC8i/ysDfB0bqLNRHZdvQMiglHmuhkuMj0ihLDSa4X6dQtUF9xo3/F
hEMmT/Rf9Fsg1Jvv8dE+77189RS3MDSBFllurWFv/lE81fiifBLkds57syDg+9amPVAbrnUrj+yiYSb67Fa4sILnwuB5k0QIXfTX
5TYKEzPCLRN7SHdiCHcyQG17Lyk7cDVeUgMm011f7eX9k3KHQo4erhcp0kv6nYFvgFWdfbtgEaOLIlX9w91SCgPRhuQs/ZD3A0TB
HSt9pkXpdNyoeKYWvigbZIWm9dW/cverUKnWsT7Yw8U8odWi4t96oqqEd8PpQuoY5+MWVDm166sbJVoj9RVr5lphm9zIsR9+eUwX
mnjL3cKJUXrPmCkx5DWy1578ta5zVFc5/nHMo3MQzMh75Z6ey+VsUAtYnUrMu7ezSAISPMdPPQA3646ZsqRFRXkvG2yyZnEEbb1R
19dem6PN6rSvvNaeIm2oSAOaFTjDJMeNefbpMjv64+XtY3xQ3Emd6jFhye/p6+MzyrowR6rZwlLiidegRCNSD+4fIVO1S58R5JvK
EWzthVJD2j4AU9RUNL7h+eJ7+5LRiq9YhVjVOTvqudm/c/+oatpnPtZYbr1wa2+e2HcLaeVWBIiSCWzCMx7IVRdRuSY1yWDIN+Ib
237znjuD9OYRxtzoqVpqd9N6ovxJcqPpnUTpCG2zFdqaFLEc0FrT0l8qw1nl/DxNZyuac9/zW6oH5ONutwkcOvqd5xZ3OrtMgEay
LDIBtctuHkTLvvGSqLT7+muW4TR094IfURYBUx9XmAGaeGgM7Amjkcw0PFQh+GNzCHASyB76EFTxuPB+EzEZsv6g4s/ZdsJc2TgU
9WoTbyrDVndlaB8GInAKLgrHGL0Zf3T44ineSU9ij3IARWZUDfZlDL4ygNrwvY15pcJDtKGpJukmIT1xpv0TQ/cY11NiFIbsVmRV
DUhNw/4edFJ8AbhyvMx0Yk53Zs5uLw+A1lkIoXRQw4Jy+3dgDn/guB7kIaRuCOVf5oKiej94paDv43VBFDdPU98uVPBmwe2aOhOx
/gCUE+0RmZJmPOmHHN9c5qixHw9sPvayYJQ2fADtGmaRm4+ajn0/ks0q8novPFEgKuty+xL2gvji/pYL2705VdgHBlVcx1c0LpHD
tPFqNJ63wtEfubz4zVN91Xe308Kl5VnjPfhWmaK4nhlk/xH7LFAPZvDWqvwA5P6pPVi6eACyXJZnVoukZZVUwYUxJ3sbyk12Nglb
Z1//vL8fXs21p5uTQhzinrVWznQTucwZq0dd1eYFd7jxgMrE2QMjkasH947kfD9vWRc17m1+sbe56MrFrOhMeE9u8nxo2XzYMikU
8b2Idc/2NqaF5z74VsIWtaKy232yn2VEva6dyfZZMZdqN58crmJlDnoP8bd2vtUPcQcm6Fd6vLKHISvLtSI55MxtU1uytszQh3H3
QFJxnQokfTmLvg+muAXGhilV9sKsxIXqJ0IP/rCLnRUSBHm4vBngWC4HABPBtXv4HmmvvD17yfvk7H3tYRdXhq19RYWDyLiE7HHr
Og8wUSl2KgOWeIxqlYqXH42VacLvnrb1jGNC0uVq7VwTomAKZ1G1GrbSv8ZBr4a1ixTZRdUrVqvFdZcFirf/WySa96zbdtU9k9be
oH6PIeMFlPpOSsqGd28xetxbrYcn1pA4SUeVC+tHW3ej0/aB9om/vMW99BVpnVZldasT+tbVGFtH8a17Ib6RRJJR9bo3ExYTnLa1
+tf3yDtdPL+jMfVU4mJKS0GS9urm3du229hzyWJxf4OCKzfhPwTc3gSRbwvFP+UNyT+tDrFFjPmLKFmwilTvBR1NhmTfog2PPfw7
iai46nKI00u5AWZ+wYTt+YzcxwC5X325RjtfZUNXx7DUKpLZvK7MpjqhV/pNrXu+5LRJdRpkLxAoiwoUdaNX6bcRi8xtbAFbVIFd
W/dLm5y9YZEKuxOssbRSmbeEN/0i8ihkyrztOpbtXQVc9/sLNteWEOo4t5Z7K76Cyq81FCs2qmQIfowMQdJIBqlscNiRZaTLts4t
9t8a8lalvWZu6ckOY1kTvs4+JQA765TJjygyRrSaoK3boFUqu9jpTESvS3cquJTRTvVGklJV24bKDt6+arQCWa8cXheDW7p3bxIL
tbOmmu+sVjGdKb2KN0VldhDbHfdS5dhRNZtZHArD2kitylC5jBkAU/m2WoBsrpCPIvdlIk2pcIGP56knAaND57DBJFzbU8lrZXKu
XSE5I3gnjQ9Ij/ZtNwMvaUJOF7o6OsIJBrUYan644Vpmty9E0deol7+LQH7UYmLgheru6kciHXWpdfNmgYUVnWVItmTg69KPZP+D
dKj0b05t1v7EBSvroyayjpT0qT+qowFLjS30VWEN9roLDdpOmDVZz1Qr4iTWH5U8kt3Ody7E3uqjZgXjVeODjx5NDbW2lzGKNzUU
9LUsFrLUvDW5X+VsUBeo3CNcB5RWFMM38utAZVsxk1z7EuS6xupe9N3W+uZr016q1QIo1cEK5vgeql86craCgHVASkVoe/TDMY/x
FngUFeugKGW8AqLQz60srJrzBYudI6IlCOOds5OaailhEp76hEeNabGbUrWTB6VyRmzYdZMkXiN7XMbZVB4XqZ1cO62JadGDu8pW
KydNxb2tM6DcK6MhfKY0hLvdS34fuqh4aqOoIOoRWlv+A1wkRkkNNxY/ObMbr66DVrGgmD5dxWzbwIK51KgKGy1iOgb77k4nxfWZ
+3+2Zt9hlH/9n/8qvHbr2zpBYWoWm0F/LQqRkuFyvmFjfghQv6LgkhdGwAneh63DOvh5hx5cHmiU270aNXen1qW3gvxbusQNM0N2
m1Og6KhGoVjXje+7PztrXk6sBuSVXv61hRx2jrzTa3eYZTrhvQ3ReBU16jcoSvSLU/4kz4R/Pj15/0xYEQbpAmH9zPwkj/z8GOQo
g9Soxaggjh3KpcCfX9A513bcveti6j79EQuPmXR6upHsp31r7Rr0cX+gjmXW0uxXn4H6yGHajZxIxDEzpeFs7B/VKzK6ebX4pp7A
RjOCebyHttQPzUBCKxNO/UZAuuDlE94uEyGBfIaAbzCUGJZ9rrO+5f76Rhd8g1OWIXzrXGWSzHjd2GiHNL3Z3/25qcqJcEMRIhjz
zVpUXN7gzE+1joplhF+rpK6A03dFjUKVuNZHLBjYw8uh/PTDcCi+hOGQQanh0DW/c8AI1cH/B7f3RPHKdAAA
"""
open("agent.py", "wb").write(gzip.decompress(base64.b64decode("".join(B.split()))))

KN = sorted(f for f in glob.glob("chevalier*.*") if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp")))
if not KN:
    raise SystemExit("❌ Image du chevalier introuvable : nomme-la chevalier.png (ou chevalier1.png, chevalier2.png…)")
MEDIA = (".jpg", ".jpeg", ".png", ".webp", ".mp4", ".mov")
if not os.listdir("fonds"):   # fonds posés à la racine du dépôt
    for f in os.listdir("."):
        if f.lower().endswith(MEDIA) and not f.startswith(("chevalier", "musique", "ma_voix")):
            shutil.move(f, "fonds/" + f)
if FONDS == "fichiers" and not os.listdir("fonds"):
    raise SystemExit("❌ Aucun arrière-plan trouvé : ajoute des images/vidéos, ou mets FONDS = \"genere\"")
if not os.path.exists("textes.txt"):
    open("textes.txt", "w", encoding="utf-8").write(TEXTES_PAR_DEFAUT)
MU = sorted(f for f in glob.glob("musique*.*") if f.lower().endswith((".mp3", ".m4a", ".wav", ".aac")))
print("   chevaliers :", KN, "| musiques :", MU, "| fonds :", len(os.listdir("fonds")))

print("3/5 Génération des vidéos (voix, sous-titres, montage)…")
KNS = ",".join(KN)
cmd = f"python agent.py --scripts textes.txt --bg fonds --knight {KNS} --knight_pos {CHEVALIER} --speed {VITESSE} --canvas {FORMAT} --accent {ACCENT} --bg_mode {FONDS} --bg_n {NB_IMAGES} --gen_engine {MOTEUR_IMAGES} --out sortie"
ref = next(iter(glob.glob("ma_voix.*")), None)
if CLONER and ref:
    cmd += f" --engine chatterbox --ref_voice {ref} --expressivity {EXPRESSIVITE} --deep 0"
else:
    cmd += f" --engine edge --voice {VOIX} --deep {GRAVE}"
if MU:
    cmd += " --music " + ",".join(MU)
sh(cmd)

print("4/5 Téléchargement…")
for f in sorted(glob.glob("sortie/*.mp4")):
    files.download(f)
print("5/5 ✅ Terminé")
